# download_com_legacy_prefinal_v7

This notebook scrapes the old European Commission DG Competition yearly antitrust pages:

`http://ec.europa.eu/competition/antitrust/closed/en/{year}.html`

It follows the same modern source layout as the other collection notebooks: raw cached source files in `data/raw/com_legacy`, manifests in `output/com_legacy`, and logs in `logs/com_legacy`. Document downloading is optional and controlled by `DOWNLOAD_FILES = False` by default.


Prefinal v6 changes: richer legacy block parsing, normalized decision categories, extra link columns, and direct full-text/OJ fallback downloads when CELEX is missing or insufficient.

Prefinal v6 fix: direct fallback links keep their original source basename (for example `iv34801_en.pdf`) instead of adding a `source_entry_id` prefix, so already-downloaded files are detected using the same naming convention as before.


Prefinal v7 fix: stronger footer/navigation filtering, PDF downloads use API request first, and old `Download is starting` rows are retryable.

In [46]:
from __future__ import annotations

import os
import re
import json
import time
import random
import hashlib
import logging
import contextlib
import sys
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag
from tqdm.auto import tqdm

from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


## 1. Paths and configuration


In [47]:
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]  # code/notebooks -> code -> project root

CODE_DIR = PROJECT_ROOT / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
SCRIPTS_DIR = CODE_DIR / "scripts"
SRC_DIR = CODE_DIR / "src"
CONFIG_DIR = PROJECT_ROOT / "config"
DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output"
LOGS_DIR = PROJECT_ROOT / "logs"

SOURCE_NAME = "com_legacy"
RAW_DATA_DIR = DATA_DIR / "raw"
COM_LEGACY_RAW_DIR = RAW_DATA_DIR / SOURCE_NAME
COM_LEGACY_YEAR_HTML_DIR = COM_LEGACY_RAW_DIR / "html_year_pages"
COM_LEGACY_FILES_DIR = COM_LEGACY_RAW_DIR / "files"
COM_LEGACY_FILE_HTML_DIR = COM_LEGACY_FILES_DIR / "html"
COM_LEGACY_FILE_PDF_DIR = COM_LEGACY_FILES_DIR / "pdfs"
COM_LEGACY_OUTPUT_DIR = OUTPUT_DIR / SOURCE_NAME
COM_LEGACY_LOGS_DIR = LOGS_DIR / SOURCE_NAME

for path in [COM_LEGACY_RAW_DIR, COM_LEGACY_YEAR_HTML_DIR, COM_LEGACY_FILES_DIR, COM_LEGACY_FILE_HTML_DIR, COM_LEGACY_FILE_PDF_DIR, COM_LEGACY_OUTPUT_DIR, COM_LEGACY_LOGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

YEAR_START = 1964
YEAR_END = 2000
YEARS = list(range(YEAR_START, YEAR_END + 1))
BASE_URL = "http://ec.europa.eu/competition/antitrust/closed/en/{year}.html"

FORCE_REFETCH_YEAR_PAGES = False
SOURCE_REQUEST_SLEEP_SECONDS = 0.15
SOURCE_REQUEST_TIMEOUT_SECONDS = 30

DOWNLOAD_FILES = True  # default: manifest only
DOWNLOAD_LIMIT = None
OVERWRITE_EXISTING_FILES = False
SAVE_EVERY = 10
DOWNLOAD_SLEEP_SECONDS = 0.3
DOWNLOAD_TIMEOUT_SECONDS = 45
PLAYWRIGHT_NAVIGATION_TIMEOUT_MS = 5000
PLAYWRIGHT_WAIT_AFTER_GOTO_MS = 0
DOWNLOAD_ATTEMPTS = [("EN", "html"), ("EN", "pdf"), ("DE", "html"), ("DE", "pdf")]

# Retry policy:
# - pending-like rows are always attempted
# - HTTP 202 rows are retryable because EUR-Lex may initially accept the request but not return content yet
# - HTTP 400/404 style failures are treated as terminal unless you manually reset them to pending
RETRY_HTTP_STATUSES = {202}
RETRY_DOWNLOAD_STATUSES = {"failed_http_202_accepted", "failed_playwright_download_event"}

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data:     {COM_LEGACY_RAW_DIR}")
print(f"Output:       {COM_LEGACY_OUTPUT_DIR}")
print(f"Logs:         {COM_LEGACY_LOGS_DIR}")


Project root: /home/edik/projects/eccjeu
Raw data:     /home/edik/projects/eccjeu/data/raw/com_legacy
Output:       /home/edik/projects/eccjeu/output/com_legacy
Logs:         /home/edik/projects/eccjeu/logs/com_legacy


## 2. Logging and save helpers


In [48]:
log_file = COM_LEGACY_LOGS_DIR / f"download_com_legacy_{datetime.now():%Y%m%d_%H%M%S}.log"
logger = logging.getLogger("download_com_legacy")
logger.setLevel(logging.INFO)
logger.handlers.clear()
file_handler = logging.FileHandler(log_file, encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)
logger.propagate = False
print(f"Log file: {log_file}")


def _prepare_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "http_status" in out.columns:
        out["http_status"] = pd.to_numeric(out["http_status"].replace("", pd.NA), errors="coerce").astype("Int64")
    if "download_success" in out.columns:
        out["download_success"] = out["download_success"].astype("boolean")
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].astype("string")
    return out


def save_dataframe(df: pd.DataFrame, csv_path: Path, parquet_path: Path | None = None) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False, encoding="utf-8")
    if parquet_path is not None:
        try:
            _prepare_for_parquet(df).to_parquet(parquet_path, index=False)
        except Exception as e:
            logger.warning("Could not save parquet file %s: %s", parquet_path, e)


Log file: /home/edik/projects/eccjeu/logs/com_legacy/download_com_legacy_20260831_145306.log


## 3. Fetch and cache the legacy yearly pages


In [49]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; ZHAW-EU-Antitrust-Research/1.0)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8",
})


def html_path_for_year(year: int) -> Path:
    return COM_LEGACY_YEAR_HTML_DIR / f"ec_competition_antitrust_closed_{year}.html"


def fetch_year_html(year: int, force: bool = False) -> dict:
    url = BASE_URL.format(year=year)
    out_path = html_path_for_year(year)
    if out_path.exists() and not force:
        return {"year": year, "url": url, "status_code": 200, "from_cache": True, "path": str(out_path), "bytes": out_path.stat().st_size, "ok": True, "error": ""}
    try:
        response = SESSION.get(url, timeout=SOURCE_REQUEST_TIMEOUT_SECONDS)
        content = response.content or b""
        ok = response.status_code == 200 and len(content) > 200
        if ok:
            out_path.write_bytes(content)
        time.sleep(SOURCE_REQUEST_SLEEP_SECONDS + random.random() * 0.1)
        return {"year": year, "url": url, "status_code": response.status_code, "from_cache": False, "path": str(out_path) if ok else "", "bytes": len(content), "ok": ok, "error": "" if ok else f"unexpected_status_or_size_{response.status_code}_{len(content)}"}
    except Exception as e:
        logger.exception("Failed to fetch year page %s", year)
        return {"year": year, "url": url, "status_code": pd.NA, "from_cache": False, "path": "", "bytes": 0, "ok": False, "error": repr(e)}

fetch_log = []
for year in tqdm(YEARS, desc="Fetching legacy yearly pages"):
    fetch_log.append(fetch_year_html(year, force=FORCE_REFETCH_YEAR_PAGES))
fetch_df = pd.DataFrame(fetch_log)
save_dataframe(fetch_df, COM_LEGACY_LOGS_DIR / "fetch_year_pages_log.csv", COM_LEGACY_LOGS_DIR / "fetch_year_pages_log.parquet")
print(fetch_df["ok"].value_counts(dropna=False))
fetch_df.head()


Fetching legacy yearly pages:   0%|          | 0/37 [00:00<?, ?it/s]

ok
True    37
Name: count, dtype: int64


,year,url,status_code,from_cache,path,bytes,ok,error
0,1964,http://ec.europa.eu/competition/antitrust/clos...,200,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,6234,True,
1,1965,http://ec.europa.eu/competition/antitrust/clos...,200,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,5101,True,
2,1966,http://ec.europa.eu/competition/antitrust/clos...,200,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,103586,True,
3,1967,http://ec.europa.eu/competition/antitrust/clos...,200,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,4195,True,
4,1968,http://ec.europa.eu/competition/antitrust/clos...,200,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,7914,True,


## 4. Parser helpers


In [50]:

DATE_RE = re.compile(r"\b\d{2}[./]\d{2}[./]\d{4}\b")
CASE_NO_RE = re.compile(r"\b([A-Z]{1,4})\s*/\s*([0-9]{1,6}[A-Z]?)\b")
CELEX_IN_TEXT_RE = re.compile(r"Celex\s+No\.\s*:\s*([0-9A-Za-z\s]+)", re.I)

# Handles rows such as "London Rubber Terminal Market Association Ltd 385D05 66 - IV/27593"
# where CELEX is visible text, not a link and not introduced by "Celex No. :".
CELEX_GENERIC_RE = re.compile(r"\b3\s*\d{2,4}\s*[A-Z]\s*\d{2}\s*\d{2}\b", re.I)


def clean_text(x: str | None) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", x.replace("\xa0", " ")).strip()


def normalize_celex(x: str | None) -> str:
    return re.sub(r"[^0-9A-Za-z]", "", x or "").upper()


def celex_to_complete(celex: str | None) -> str:
    """Convert old shortened CELEX forms such as 385D0566 to 31985D0566."""
    c = normalize_celex(celex)
    if not c:
        return ""
    if re.fullmatch(r"3\d{4}[A-Z]\d{4}", c):
        return c
    m = re.fullmatch(r"3(\d{2})([A-Z])(\d{4})", c)
    if m:
        yy, letter, number = m.groups()
        yyyy = f"19{yy}" if int(yy) >= 50 else f"20{yy}"
        return f"3{yyyy}{letter}{number}"
    return c


def parse_date_to_iso(date_str: str) -> str:
    """Normalize legacy page dates to the display format dd.mm.yyyy.

    The source pages use dd.mm.yyyy or dd/mm/yyyy.  We keep the
    canonical manifest date in the classic European form because that is
    the format used in the manual review workflow.
    """
    if not date_str:
        return ""
    sep = "." if "." in date_str else "/"
    dd, mm, yyyy = date_str.split(sep)
    return f"{dd.zfill(2)}.{mm.zfill(2)}.{yyyy}"


def year_from_display_date(date_str: str):
    """Return the year from a dd.mm.yyyy date; pd.NA when absent."""
    if not date_str:
        return pd.NA
    return int(str(date_str).split(".")[-1])


def display_date_sort_key(date_str: str):
    """Parse dd.mm.yyyy for chronological sorting without changing exported columns."""
    return pd.to_datetime(date_str, format="%d.%m.%Y", errors="coerce")


def extract_celex_from_text(text: str) -> tuple[list[str], list[str]]:
    """Extract CELEX values from plain text, including old broken forms like '385D05 66'."""
    text = clean_text(text)
    values = []

    # Pattern after "Celex No. :"
    for m in CELEX_IN_TEXT_RE.finditer(text):
        raw = m.group(1)
        raw = re.split(r"\s+-\s+|\s+IV\s*/|\s+Official Journal\s*:", raw, flags=re.I)[0]
        c = normalize_celex(raw)
        if re.fullmatch(r"3\d{2,4}[A-Z]\d{4}", c):
            values.append(c)

    # Generic visible CELEX pattern anywhere in the text
    for m in CELEX_GENERIC_RE.finditer(text):
        c = normalize_celex(m.group(0))
        if re.fullmatch(r"3\d{2,4}[A-Z]\d{4}", c):
            values.append(c)

    values = list(dict.fromkeys(values))
    complete = list(dict.fromkeys([celex_to_complete(c) for c in values if c]))
    return values, complete


def extract_links_from_fragment(fragment_html: str, source_url: str) -> list[dict]:
    """Return all links in a logical source line/block."""
    soup = BeautifulSoup(fragment_html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        links.append({
            "text": clean_text(a.get_text(" ", strip=True)),
            "href": href,
            "url": urljoin(source_url, href),
        })
    return links


def extract_celex_from_links(links: list[dict]) -> tuple[list[str], list[str], list[str]]:
    celex_raw, celex_urls = [], []
    for link in links or []:
        href = link.get("href", "") or ""
        url = link.get("url", "") or ""
        text = link.get("text", "") or ""
        haystack = f"{href} {url} {text}"
        if "CELEXnumdoc" in haystack or "numdoc=" in haystack or "CELEX:" in haystack.upper():
            if url:
                celex_urls.append(url)
            parsed = urlparse(href)
            qs = parse_qs(parsed.query)
            for v in (qs.get("numdoc") or qs.get("uri") or []):
                c = normalize_celex(v.replace("CELEX:", ""))
                if c:
                    celex_raw.append(c)
            for m in re.finditer(r"(?:numdoc=|CELEX:)([0-9A-Za-z]+)", href, flags=re.I):
                c = normalize_celex(m.group(1))
                if c:
                    celex_raw.append(c)
            visible = normalize_celex(text)
            if re.fullmatch(r"3\d{2,4}[A-Z]\d{4}", visible):
                celex_raw.append(visible)

    celex_raw = list(dict.fromkeys(celex_raw))
    celex_complete = list(dict.fromkeys([celex_to_complete(c) for c in celex_raw if c]))
    celex_urls = list(dict.fromkeys(celex_urls))
    return celex_raw, celex_complete, celex_urls


def extract_case_numbers(text: str) -> list[str]:
    return list(dict.fromkeys([f"{prefix}/{num}" for prefix, num in CASE_NO_RE.findall(text or "")]))


def extract_official_journal(text: str) -> dict:
    result = {
        "official_journal_raw": "",
        "oj_series": "",
        "oj_issue": "",
        "oj_date": "",
        "oj_page_raw": "",
    }
    if "Official Journal" not in (text or ""):
        return result

    m = re.search(
        r"Official Journal\s*:\s*(.*?)(?=\s+Celex\s+No\.|\s+-\s+\b[A-Z]{1,4}\s*/\s*\d|$)",
        text,
        flags=re.I,
    )
    if not m:
        return result

    raw = clean_text(m.group(1)).strip(" -")
    result["official_journal_raw"] = raw

    m2 = re.search(
        r"\b(?P<series>[LC])\s*(?P<issue>\d+)?\s*-\s*(?P<date>\d{1,2}[./]\d{1,2}[./]\d{4})?(?:\s+Page\s*:\s*(?P<page>[0-9\-– ]+))?",
        raw,
        flags=re.I,
    )
    if m2:
        result["oj_series"] = (m2.group("series") or "").upper()
        result["oj_issue"] = m2.group("issue") or ""
        result["oj_date"] = parse_date_to_iso(m2.group("date") or "") if m2.group("date") else ""
        result["oj_page_raw"] = clean_text(m2.group("page") or "")

    return result


def extract_press_release_urls(links: list[dict]) -> list[str]:
    urls = []
    for link in links or []:
        label = clean_text(link.get("text", "")).lower()
        href = (link.get("href", "") or "").lower()
        url = link.get("url", "") or ""
        if "press" in label or "rapid" in href:
            urls.append(url)
    return list(dict.fromkeys(urls))


def split_html_into_logical_lines(html: str, source_url: str) -> list[dict]:
    """
    Split the legacy HTML source into logical visual lines.

    This is deliberately line/block based: each source line is parsed on its own,
    and the following Official Journal line is attached only to the immediately
    preceding case. We no longer borrow metadata from nearby linked rows.
    """
    # Remove scripts/styles and normalize line breaks.
    html_clean = re.sub(r"<script\b.*?</script>", "", html, flags=re.I | re.S)
    html_clean = re.sub(r"<style\b.*?</style>", "", html_clean, flags=re.I | re.S)
    # Most pages use <br> as row separators.
    parts = re.split(r"<br\s*/?>", html_clean, flags=re.I)

    rows = []
    for i, fragment in enumerate(parts):
        soup = BeautifulSoup(fragment, "html.parser")
        text = clean_text(soup.get_text(" ", strip=True))
        if not text:
            continue
        rows.append({
            "html_line_index": i,
            "source_line_text": text,
            "source_line_html": fragment.strip(),
            "source_line_links_list": extract_links_from_fragment(fragment, source_url),
        })
    return rows


def is_navigation_or_header_line(text: str) -> bool:
    low = clean_text(text).lower()
    if not low:
        return True
    skip_bits = [
        "important legal notice",
        "the information on this site",
        "decisions art.",
        "last page update",
        "index of antitrust formal decisions",
        "previous",
        "next",
        "search",
        "top",
        "europa",
        "competition",
        "european commission",
    ]
    return any(bit in low for bit in skip_bits)


def is_detail_line(text: str) -> bool:
    low = clean_text(text).lower()
    return low.startswith("official journal") or " official journal" in low or low.startswith("celex no")


def looks_like_case_line(text: str) -> bool:
    """Conservative enough to avoid headers, permissive enough to retain incomplete rows."""
    text = clean_text(text)
    if is_navigation_or_header_line(text) or is_detail_line(text):
        return False
    return bool(DATE_RE.search(text) or CELEX_GENERIC_RE.search(text) or CASE_NO_RE.search(text) or len(text) >= 3)


def strip_metadata_from_decision_type(text: str) -> str:
    cut_positions = []
    for pat in [
        r"Official Journal\s*:",
        r"Celex\s+No\.\s*:",
        r"\b3\s*\d{2,4}\s*[A-Z]\s*\d{2}\s*\d{2}\b",
        r"\s+-\s+\b[A-Z]{1,4}\s*/\s*\d{1,6}",
        r"\b[A-Z]{1,4}\s*/\s*\d{1,6}\b",
        r"\s+Press Release\b",
    ]:
        m = re.search(pat, text, flags=re.I)
        if m:
            cut_positions.append(m.start())
    end = min(cut_positions) if cut_positions else len(text)
    return clean_text(text[:end]).strip(" -")


def parse_title_date_type_from_line(case_line_text: str) -> dict:
    text = clean_text(case_line_text)
    m_date = DATE_RE.search(text)

    if m_date:
        title = clean_text(text[:m_date.start()])
        decision_date_raw = m_date.group(0)
        decision_date = parse_date_to_iso(decision_date_raw)
        decision_type = strip_metadata_from_decision_type(text[m_date.end():])
        return {
            "title": title,
            "decision_date_raw": decision_date_raw,
            "decision_date": decision_date,
            "decision_type": decision_type,
        }

    # No date: take title up to the first metadata-looking marker.
    cut_positions = []
    for pat in [
        r"\b3\s*\d{2,4}\s*[A-Z]\s*\d{2}\s*\d{2}\b",
        r"\b[A-Z]{1,4}\s*/\s*\d{1,6}",
        r"Official Journal\s*:",
        r"Celex\s+No\.\s*:",
    ]:
        m = re.search(pat, text, flags=re.I)
        if m:
            cut_positions.append(m.start())
    end = min(cut_positions) if cut_positions else len(text)
    title = clean_text(text[:end]).strip(" -") or text
    return {
        "title": title,
        "decision_date_raw": "",
        "decision_date": "",
        "decision_type": "",
    }


def parse_case_line_record(year: int, source_url: str, case_line: dict, detail_lines: list[dict]) -> dict | None:
    case_text = clean_text(case_line.get("source_line_text", ""))
    if not looks_like_case_line(case_text):
        return None

    detail_texts = [clean_text(x.get("source_line_text", "")) for x in detail_lines]
    combined_text = clean_text(" ".join([case_text, *detail_texts]))

    all_links = []
    all_links.extend(case_line.get("source_line_links_list", []) or [])
    for detail in detail_lines:
        all_links.extend(detail.get("source_line_links_list", []) or [])

    text_celex_raw, text_celex_complete = extract_celex_from_text(combined_text)
    link_celex_raw, link_celex_complete, celex_urls = extract_celex_from_links(all_links)
    celex_raw = list(dict.fromkeys([*link_celex_raw, *text_celex_raw]))
    celex_complete = list(dict.fromkeys([*link_celex_complete, *text_celex_complete]))
    case_numbers = extract_case_numbers(combined_text)
    oj = extract_official_journal(combined_text)
    press_release_urls = extract_press_release_urls(all_links)
    title_date_type = parse_title_date_type_from_line(case_text)

    line_index = case_line.get("html_line_index", "")
    source_entry_id = f"com_legacy_{year}_line_{line_index}"
    row_hash = hashlib.sha1(
        f"{source_url}|{line_index}|{combined_text}".encode("utf-8")
    ).hexdigest()[:12]

    parse_mode = "line_with_detail" if detail_lines else "line_only"
    if not title_date_type["decision_date"]:
        parse_mode += "_no_date"
    if not detail_lines and (celex_complete or case_numbers):
        parse_mode += "_inline_metadata"

    return {
        "source_entry_id": source_entry_id,
        "row_hash": row_hash,
        "source_name": SOURCE_NAME,
        "source_url": source_url,
        "source_year": year,
        "source_anchor": "",
        "source_line_index": line_index,
        "detail_line_indices": "; ".join(str(x.get("html_line_index", "")) for x in detail_lines),
        "parse_mode": parse_mode,
        **title_date_type,
        "decision_year": year_from_display_date(title_date_type["decision_date"]),
        **oj,
        "celex_raw_list": celex_raw,
        "celex_complete_list": celex_complete,
        "celex_raw": "; ".join(celex_raw),
        "celex_complete": "; ".join(celex_complete),
        "celex_primary": celex_complete[0] if celex_complete else "",
        "celex_url_list": celex_urls,
        "celex_urls": "; ".join(celex_urls),
        "case_number_list": case_numbers,
        "case_numbers": "; ".join(case_numbers),
        "case_number_primary": case_numbers[0] if case_numbers else "",
        "press_release_url_list": press_release_urls,
        "press_release_urls": "; ".join(press_release_urls),
        "source_line_text": case_text,
        "detail_line_text": " | ".join(detail_texts),
        "details_text_raw": combined_text,
        "source_line_html": case_line.get("source_line_html", ""),
        "detail_line_html": "\n---DETAIL-LINE---\n".join(x.get("source_line_html", "") for x in detail_lines),
        "all_links_list": all_links,
        "source_line_links_list": case_line.get("source_line_links_list", []) or [],
        "detail_line_links_list": [x.get("source_line_links_list", []) or [] for x in detail_lines],
        "all_links_json": json.dumps(all_links, ensure_ascii=False),
        "has_decision_date": bool(title_date_type["decision_date"]),
        "has_celex": bool(celex_complete),
        "has_official_journal": bool(oj["official_journal_raw"]),
        "has_case_number": bool(case_numbers),
        "needs_manual_review": not bool(title_date_type["decision_date"]) or not bool(celex_complete),
    }


def parse_year_page_lines(year: int, source_url: str, html: str) -> tuple[list[dict], list[dict]]:
    """
    Parse every case from the actual logical lines of the HTML source.

    Each case line becomes one record. The next adjacent detail line is attached only
    if it is explicitly a detail line (Official Journal / Celex). Missing values are
    left blank instead of being borrowed from the next case.
    """
    logical_lines = split_html_into_logical_lines(html, source_url)
    records = []
    debug = []

    current_case = None
    current_details = []

    def flush_current():
        nonlocal current_case, current_details
        if current_case is None:
            return
        rec = parse_case_line_record(year, source_url, current_case, current_details)
        if rec is not None:
            records.append(rec)
        current_case = None
        current_details = []

    for line in logical_lines:
        text = clean_text(line.get("source_line_text", ""))
        debug.append({"year": year, "html_line_index": line.get("html_line_index"), "text": text})

        if is_navigation_or_header_line(text):
            continue

        if is_detail_line(text):
            if current_case is not None:
                current_details.append(line)
            # Orphan detail lines are ignored because they cannot be assigned safely.
            continue

        if looks_like_case_line(text):
            flush_current()
            current_case = line
            current_details = []
            continue

    flush_current()
    return records, debug


### Date parsing convention

Legacy Commission pages expose dates as `dd.mm.yyyy` or occasionally `dd/mm/yyyy`. This notebook now normalizes parsed dates to the classic European display format `dd.mm.yyyy` in the exported manifest columns, including `decision_date` and `oj_date`.

Downstream logic that needs the year no longer assumes ISO format (`yyyy-mm-dd`). It extracts the year from the final component of `dd.mm.yyyy`, and sorting uses a temporary datetime sort key so the exported date format remains unchanged.


In [51]:

# ---- V3 parser extensions: attach multi-line link rows and classify non-CELEX links ----

def is_detail_line(text: str) -> bool:
    """Rows that belong to the previous case/decision entry, not standalone cases."""
    low = clean_text(text).lower()
    if not low:
        return False
    detail_markers = [
        "official journal",
        "celex no",
        "press release",
        "full text",
        "text in pdf format",
        "text in pdf",
        "text pdf",
    ]
    # Some rows are simply "Text" plus a PDF/HTML icon link.
    if low in {"text", "text pdf", "pdf", "html"}:
        return True
    return any(marker in low for marker in detail_markers)


def looks_like_case_line(text: str) -> bool:
    """Conservative case-line detector; detail rows are not allowed to become cases."""
    text = clean_text(text)
    if is_navigation_or_header_line(text) or is_detail_line(text):
        return False
    return bool(DATE_RE.search(text) or CELEX_GENERIC_RE.search(text) or CASE_NO_RE.search(text) or len(text) >= 3)


def _dedupe_keep_order(values: list[str]) -> list[str]:
    out = []
    seen = set()
    for value in values or []:
        value = str(value or "").strip()
        if not value or value in seen:
            continue
        seen.add(value)
        out.append(value)
    return out


def _link_is_celex(link: dict) -> bool:
    haystack = f"{link.get('href','')} {link.get('url','')} {link.get('text','')}".upper()
    return "CELEXNUMDOC" in haystack or "NUMDOC=" in haystack or "CELEX:" in haystack


def classify_case_links(links: list[dict]) -> dict:
    """
    Classify all links attached to one legacy case block.

    The legacy pages often have useful non-CELEX links below the case line:
    Full text, Text in PDF Format, Press Release, and occasionally an OJ EUR-Lex link.
    We keep them on the case row instead of making a link-level source manifest.
    """
    buckets = {
        "full_text_links_list": [],
        "pdf_links_list": [],
        "html_links_list": [],
        "press_release_links_list": [],
        "oj_external_links_list": [],
        "other_links_list": [],
    }

    for link in links or []:
        url = str(link.get("url", "") or "").strip()
        href = str(link.get("href", "") or "").strip()
        label = clean_text(link.get("text", ""))
        context = clean_text(link.get("line_text", ""))
        haystack = f"{label} {context} {href} {url}".lower()
        if not url:
            continue

        path = urlparse(url).path.lower()
        is_pdf = path.endswith(".pdf") or ".pdf" in path
        is_html = path.endswith(".html") or path.endswith(".htm") or ".html" in path or ".htm" in path
        is_press = "press release" in haystack or "rapid" in haystack or "p_action.gettxt" in haystack
        is_oj = "official journal" in haystack or "uriserv:oj" in haystack or "toc=oj" in haystack or "oj.l_" in haystack
        is_full_text = (
            "full text" in haystack
            or "text in pdf" in haystack
            or context.lower().strip() in {"text", "full text", "text in pdf format"}
            or (is_pdf and not is_press and not is_oj and not _link_is_celex(link))
        )

        if is_pdf:
            buckets["pdf_links_list"].append(url)
        if is_html:
            buckets["html_links_list"].append(url)
        if is_press:
            buckets["press_release_links_list"].append(url)
        elif _link_is_celex(link):
            # CELEX links are stored separately by extract_celex_from_links.
            pass
        elif is_oj:
            buckets["oj_external_links_list"].append(url)
        elif is_full_text:
            buckets["full_text_links_list"].append(url)
        else:
            buckets["other_links_list"].append(url)

    for key in buckets:
        buckets[key] = _dedupe_keep_order(buckets[key])
    return buckets


def parse_case_line_record(year: int, source_url: str, case_line: dict, detail_lines: list[dict]) -> dict | None:
    case_text = clean_text(case_line.get("source_line_text", ""))
    if not looks_like_case_line(case_text):
        return None

    detail_texts = [clean_text(x.get("source_line_text", "")) for x in detail_lines]
    combined_text = clean_text(" ".join([case_text, *detail_texts]))

    all_links = []
    for source in [case_line, *detail_lines]:
        line_text = clean_text(source.get("source_line_text", ""))
        for link in source.get("source_line_links_list", []) or []:
            link_copy = dict(link)
            link_copy["line_text"] = line_text
            all_links.append(link_copy)

    text_celex_raw, text_celex_complete = extract_celex_from_text(combined_text)
    link_celex_raw, link_celex_complete, celex_urls = extract_celex_from_links(all_links)
    celex_raw = _dedupe_keep_order([*link_celex_raw, *text_celex_raw])
    celex_complete = _dedupe_keep_order([*link_celex_complete, *text_celex_complete])
    case_numbers = extract_case_numbers(combined_text)
    oj = extract_official_journal(combined_text)
    link_buckets = classify_case_links(all_links)
    title_date_type = parse_title_date_type_from_line(case_text)

    line_index = case_line.get("html_line_index", "")
    source_entry_id = f"com_legacy_{year}_line_{line_index}"
    row_hash = hashlib.sha1(
        f"{source_url}|{line_index}|{combined_text}".encode("utf-8")
    ).hexdigest()[:12]

    parse_mode = "line_with_detail" if detail_lines else "line_only"
    if not title_date_type["decision_date"]:
        parse_mode += "_no_date"
    if not detail_lines and (celex_complete or case_numbers):
        parse_mode += "_inline_metadata"
    if link_buckets["full_text_links_list"] or link_buckets["oj_external_links_list"]:
        parse_mode += "_with_external_links"

    direct_doc_links = _dedupe_keep_order([
        *link_buckets["full_text_links_list"],
        *link_buckets["oj_external_links_list"],
    ])
    all_download_links = _dedupe_keep_order([*celex_urls, *direct_doc_links])

    return {
        "source_entry_id": source_entry_id,
        "row_hash": row_hash,
        "source_name": SOURCE_NAME,
        "source_url": source_url,
        "source_year": year,
        "source_anchor": "",
        "source_line_index": line_index,
        "detail_line_indices": "; ".join(str(x.get("html_line_index", "")) for x in detail_lines),
        "parse_mode": parse_mode,
        **title_date_type,
        "decision_year": year_from_display_date(title_date_type["decision_date"]),
        **oj,
        "celex_raw_list": celex_raw,
        "celex_complete_list": celex_complete,
        "celex_raw": "; ".join(celex_raw),
        "celex_complete": "; ".join(celex_complete),
        "celex_primary": celex_complete[0] if celex_complete else "",
        "celex_url_list": celex_urls,
        "celex_urls": "; ".join(celex_urls),
        "case_number_list": case_numbers,
        "case_numbers": "; ".join(case_numbers),
        "case_number_primary": case_numbers[0] if case_numbers else "",
        **link_buckets,
        "full_text_links": "; ".join(link_buckets["full_text_links_list"]),
        "pdf_links": "; ".join(link_buckets["pdf_links_list"]),
        "html_links": "; ".join(link_buckets["html_links_list"]),
        "press_release_links": "; ".join(link_buckets["press_release_links_list"]),
        "oj_external_links": "; ".join(link_buckets["oj_external_links_list"]),
        "other_links": "; ".join(link_buckets["other_links_list"]),
        "download_candidate_links_list": all_download_links,
        "download_candidate_links": "; ".join(all_download_links),
        "preferred_download_source": "celex" if celex_complete else ("full_text" if link_buckets["full_text_links_list"] else ("oj_external" if link_buckets["oj_external_links_list"] else "none")),
        "source_line_text": case_text,
        "detail_line_text": " | ".join(detail_texts),
        "details_text_raw": combined_text,
        "source_line_html": case_line.get("source_line_html", ""),
        "detail_line_html": "\n---DETAIL-LINE---\n".join(x.get("source_line_html", "") for x in detail_lines),
        "all_links_list": all_links,
        "source_line_links_list": case_line.get("source_line_links_list", []) or [],
        "detail_line_links_list": [x.get("source_line_links_list", []) or [] for x in detail_lines],
        "all_links_json": json.dumps(all_links, ensure_ascii=False),
        "has_decision_date": bool(title_date_type["decision_date"]),
        "has_celex": bool(celex_complete),
        "has_official_journal": bool(oj["official_journal_raw"]),
        "has_case_number": bool(case_numbers),
        "has_full_text_link": bool(link_buckets["full_text_links_list"]),
        "has_oj_external_link": bool(link_buckets["oj_external_links_list"]),
        "has_press_release_link": bool(link_buckets["press_release_links_list"]),
        "has_any_download_link": bool(celex_complete or direct_doc_links),
        "needs_manual_review": not bool(title_date_type["decision_date"]),
    }


def parse_year_page_lines(year: int, source_url: str, html: str) -> tuple[list[dict], list[dict]]:
    """
    Parse case blocks from the legacy HTML source.

    V3 attaches all adjacent detail/link lines to the previous case until the next
    real case line starts. This fixes rows such as Frankfurt Airport 1998, where
    the decision has no CELEX but does have a following full-text PDF link.
    """
    logical_lines = split_html_into_logical_lines(html, source_url)
    records = []
    debug = []

    current_case = None
    current_details = []

    def flush_current():
        nonlocal current_case, current_details
        if current_case is None:
            return
        rec = parse_case_line_record(year, source_url, current_case, current_details)
        if rec is not None:
            records.append(rec)
        current_case = None
        current_details = []

    for line in logical_lines:
        text = clean_text(line.get("source_line_text", ""))
        debug.append({"year": year, "html_line_index": line.get("html_line_index"), "text": text})

        if is_navigation_or_header_line(text):
            continue

        if current_case is not None and is_detail_line(text):
            current_details.append(line)
            continue

        if looks_like_case_line(text):
            flush_current()
            current_case = line
            current_details = []
            continue

        # If we are inside a case and a non-case/non-navigation line contains links,
        # keep it as a possible detail line instead of losing useful PDFs/OJ links.
        if current_case is not None and (line.get("source_line_links_list") or []):
            current_details.append(line)
            continue

    flush_current()
    return records, debug


## 5. Parse cached yearly pages into `web_scrape_manifest`


In [52]:

records, parse_log, line_debug_records = [], [], []
for year in tqdm(YEARS, desc="Parsing legacy yearly pages"):
    path = html_path_for_year(year)
    source_url = BASE_URL.format(year=year)
    if not path.exists():
        parse_log.append({"year": year, "ok": False, "n_records": 0, "error": "missing_html"})
        continue
    html = path.read_text(encoding="latin-1", errors="replace")
    n_before = len(records)
    year_records, year_line_debug = parse_year_page_lines(year, source_url, html)
    records.extend(year_records)
    line_debug_records.extend(year_line_debug)
    parse_log.append({"year": year, "ok": True, "n_records": len(records) - n_before, "error": ""})

parse_df = pd.DataFrame(parse_log)
line_debug_df = pd.DataFrame(line_debug_records)
save_dataframe(parse_df, COM_LEGACY_LOGS_DIR / "parse_year_pages_log.csv", COM_LEGACY_LOGS_DIR / "parse_year_pages_log.parquet")
save_dataframe(line_debug_df, COM_LEGACY_LOGS_DIR / "parse_year_page_lines_debug.csv", COM_LEGACY_LOGS_DIR / "parse_year_page_lines_debug.parquet")
web_scrape_manifest = pd.DataFrame(records)
print(f"Parsed rows: {len(web_scrape_manifest):,}")
parse_df.tail()


Parsing legacy yearly pages:   0%|          | 0/37 [00:00<?, ?it/s]

Parsed rows: 561


,year,ok,n_records,error
32,1996,True,22,
33,1997,True,21,
34,1998,True,43,
35,1999,True,39,
36,2000,True,2,


## 6. Save `web_scrape_manifest`


In [53]:
if web_scrape_manifest.empty:
    raise RuntimeError("No records parsed. Check fetch logs and source page availability.")

# Defensive cleanup: drop navigation/footer rows accidentally parsed as cases.
# Example bad row seen on older pages:
# "[ Index of older Antitrust Cases ] - [ 1967 ]".
def _is_legacy_navigation_row(row: pd.Series) -> bool:
    # Be deliberately broad here: this source is messy ACCESS97 HTML, and
    # footer/navigation fragments can leak into the parser as if they were case rows.
    haystack = " ".join(str(v or "") for v in row.to_dict().values()).replace("\xa0", " ")
    haystack = re.sub(r"\s+", " ", haystack).strip().lower()
    title = re.sub(r"\s+", " ", str(row.get("title", "") or "").replace("\xa0", " ")).strip().lower()

    nav_phrases = [
        "index of older antitrust cases",
        "index of antitrust formal decisions",
        "formal decisions by year",
        "previous",
        "next",
    ]

    title_is_nav = (
        "index of older antitrust cases" in title
        or "index of antitrust formal decisions" in title
        or ("index of" in title and "antitrust" in title)
    )

    # Drop explicit footer/index titles regardless of any other fields.
    if title_is_nav:
        return True

    # Also drop rows that only consist of footer/navigation text and have no real decision metadata.
    has_decision_metadata = any(str(row.get(c, "") or "").strip() for c in [
        "decision_date", "decision_date_raw", "decision_type", "case_numbers",
        "case_number_primary", "celex_primary", "celex_complete"
    ])
    looks_like_footer = any(p in haystack for p in nav_phrases) and not has_decision_metadata
    return looks_like_footer

before_nav_filter = len(web_scrape_manifest)
web_scrape_manifest = web_scrape_manifest[~web_scrape_manifest.apply(_is_legacy_navigation_row, axis=1)].copy()
removed_nav_rows = before_nav_filter - len(web_scrape_manifest)
if removed_nav_rows:
    print(f"Removed {removed_nav_rows} navigation/footer rows from web_scrape_manifest")

web_scrape_manifest = (
    web_scrape_manifest
    .assign(_decision_date_sort=web_scrape_manifest["decision_date"].apply(display_date_sort_key))
    .sort_values(["source_year", "_decision_date_sort", "source_anchor"], ascending=[True, False, True])
    .drop(columns=["_decision_date_sort"])
    .reset_index(drop=True)
)
list_cols = [c for c in web_scrape_manifest.columns if c.endswith("_list")]
web_export = web_scrape_manifest.copy()
for col in list_cols:
    web_export[col + "_json"] = web_export[col].apply(lambda x: json.dumps(x if isinstance(x, list) else [], ensure_ascii=False))
web_export = web_export.drop(columns=list_cols)

WEB_SCRAPE_MANIFEST_CSV = COM_LEGACY_OUTPUT_DIR / "web_scrape_manifest.csv"
WEB_SCRAPE_MANIFEST_PARQUET = COM_LEGACY_OUTPUT_DIR / "web_scrape_manifest.parquet"
WEB_SCRAPE_MANIFEST_JSONL = COM_LEGACY_OUTPUT_DIR / "web_scrape_manifest.jsonl"
save_dataframe(web_export, WEB_SCRAPE_MANIFEST_CSV, WEB_SCRAPE_MANIFEST_PARQUET)
with WEB_SCRAPE_MANIFEST_JSONL.open("w", encoding="utf-8") as f:
    for row in web_export.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Saved CSV: {WEB_SCRAPE_MANIFEST_CSV}")
web_export.head()


Removed 1 navigation/footer rows from web_scrape_manifest
Saved CSV: /home/edik/projects/eccjeu/output/com_legacy/web_scrape_manifest.csv


,source_entry_id,row_hash,source_name,source_url,source_year,source_anchor,source_line_index,detail_line_indices,parse_mode,title,...,full_text_links_list_json,pdf_links_list_json,html_links_list_json,press_release_links_list_json,oj_external_links_list_json,other_links_list_json,download_candidate_links_list_json,all_links_list_json,source_line_links_list_json,detail_line_links_list_json
0,com_legacy_1964_line_2,b6c4732b6f2e,com_legacy,http://ec.europa.eu/competition/antitrust/clos...,1964,,2,3,line_with_detail,Deca,...,[],[],[],[],[],[],"[""http://eur-lex.europa.eu/smartapi/cgi/sga_do...","[{""text"": ""364D05 99"", ""href"": ""http://eur-lex...",[],"[[{""text"": ""364D05 99"", ""href"": ""http://eur-le..."
1,com_legacy_1964_line_5,af45539d4a99,com_legacy,http://ec.europa.eu/competition/antitrust/clos...,1964,,5,6,line_with_detail,Grundig-Consten,...,[],[],[],[],[],[],"[""http://eur-lex.europa.eu/smartapi/cgi/sga_do...","[{""text"": ""364D05 66"", ""href"": ""http://eur-lex...",[],"[[{""text"": ""364D05 66"", ""href"": ""http://eur-le..."
2,com_legacy_1964_line_8,25044fb3567a,com_legacy,http://ec.europa.eu/competition/antitrust/clos...,1964,,8,9,line_with_detail,Nicholas Freres + Vitapro,...,[],[],[],[],[],[],"[""http://eur-lex.europa.eu/smartapi/cgi/sga_do...","[{""text"": ""364D05 02"", ""href"": ""http://eur-lex...",[],"[[{""text"": ""364D05 02"", ""href"": ""http://eur-le..."
3,com_legacy_1964_line_11,36c56c3a0db5,com_legacy,http://ec.europa.eu/competition/antitrust/clos...,1964,,11,12,line_with_detail,Bendix + Mertens and Straat,...,[],[],[],[],[],[],"[""http://eur-lex.europa.eu/smartapi/cgi/sga_do...","[{""text"": ""364D03 44"", ""href"": ""http://eur-lex...",[],"[[{""text"": ""364D03 44"", ""href"": ""http://eur-le..."
4,com_legacy_1964_line_14,399c3628d933,com_legacy,http://ec.europa.eu/competition/antitrust/clos...,1964,,14,15,line_with_detail_with_external_links,Grosfillex + Fillistorf,...,[],[],"[""http://ec.europa.eu/competition/search.html""...",[],"[""http://ec.europa.eu/competition/search.html""...",[],"[""http://eur-lex.europa.eu/smartapi/cgi/sga_do...","[{""text"": ""364D02 33"", ""href"": ""http://eur-lex...",[],"[[{""text"": ""364D02 33"", ""href"": ""http://eur-le..."


## 7. Quality checks


In [54]:
summary = web_export.groupby("source_year").agg(
    rows=("source_entry_id", "count"),
    with_celex=("has_celex", "sum"),
    with_oj=("has_official_journal", "sum"),
    with_case_no=("has_case_number", "sum"),
).reset_index()
summary["missing_celex"] = summary["rows"] - summary["with_celex"]
summary["missing_oj"] = summary["rows"] - summary["with_oj"]
summary.tail(15)


,source_year,rows,with_celex,with_oj,with_case_no,missing_celex,missing_oj
21,1986,21,21,21,21,0,0
22,1987,15,15,15,15,0,0
23,1988,24,24,24,24,0,0
24,1989,14,10,11,11,4,3
25,1990,16,15,15,16,1,1
26,1991,16,14,14,16,2,2
27,1992,32,26,26,29,6,6
28,1993,13,8,8,13,5,5
29,1994,27,23,23,27,4,4
30,1995,12,4,4,12,8,8


## 7.5 Normalize legacy decision categories

This keeps the raw legacy `decision_type`, but adds an Antitrust-Register-style normalized category that can later be aligned with the modern register labels.


In [55]:

def normalize_legacy_decision_type(value: object) -> dict:
    """
    Translate legacy DG COMP decision_type strings into a controlled vocabulary
    close to the Antitrust Register labels.

    The raw legacy field is preserved as decision_type. These columns are only a
    normalization layer for matching, filtering, and later case-manifest work.
    """
    raw = "" if pd.isna(value) else str(value).strip()
    low = raw.lower()

    if not raw:
        return {
            "normalized_decision_type": "Unknown",
            "decision_family": "Unknown",
            "keep_case_candidate": pd.NA,
            "normalization_notes": "missing legacy decision_type",
        }

    # Most specific / procedural-ish cases first.
    if "rejection of complaint" in low:
        return {
            "normalized_decision_type": "Rejection of Complaint Decision",
            "decision_family": "Rejection / Complaint Closure",
            "keep_case_candidate": False,
            "normalization_notes": "legacy rejection of complaint decision; excluded from keep universe by project rule",
        }

    if "interim measures" in low or "not to grant interim measures" in low:
        return {
            "normalized_decision_type": "Interim Measures Decision",
            "decision_family": "Interim / Procedural Decision",
            "keep_case_candidate": True,
            "normalization_notes": "legacy interim-measures wording",
        }

    if "amendment to decision" in low or low.startswith("amendment"):
        return {
            "normalized_decision_type": "Amending Decision",
            "decision_family": "Amendment",
            "keep_case_candidate": True,
            "normalization_notes": "legacy amendment to a decision",
        }

    # Old Article 81/82 substantive outcomes.
    has_infringement = "infringement" in low or "infringment" in low or "prohibition" in low
    has_exemption = "exemption" in low or "non-opposition" in low
    has_negative_clearance = "negative clearance" in low

    if has_infringement and not (has_exemption or has_negative_clearance):
        return {
            "normalized_decision_type": "Prohibition Decision",
            "decision_family": "Final Decision",
            "keep_case_candidate": True,
            "normalization_notes": "legacy infringement/prohibition wording",
        }

    if has_exemption and not (has_infringement or has_negative_clearance):
        return {
            "normalized_decision_type": "Exemption Decision",
            "decision_family": "Final Decision",
            "keep_case_candidate": True,
            "normalization_notes": "legacy exemption/non-opposition wording",
        }

    if has_negative_clearance and not (has_infringement or has_exemption):
        return {
            "normalized_decision_type": "Negative Clearance Decision",
            "decision_family": "Final Decision",
            "keep_case_candidate": True,
            "normalization_notes": "legacy negative-clearance wording",
        }

    if has_infringement and (has_exemption or has_negative_clearance):
        return {
            "normalized_decision_type": "Mixed Final Decision",
            "decision_family": "Final Decision",
            "keep_case_candidate": True,
            "normalization_notes": "legacy row combines infringement with exemption/negative clearance",
        }

    if has_exemption and has_negative_clearance:
        return {
            "normalized_decision_type": "Mixed Exemption / Negative Clearance Decision",
            "decision_family": "Final Decision",
            "keep_case_candidate": True,
            "normalization_notes": "legacy row combines exemption and negative clearance",
        }

    # Regulation 17 / procedural fines / information requests.
    if "art.11(5)" in low or "art. 11(5)" in low:
        return {
            "normalized_decision_type": "Procedural Decision",
            "decision_family": "Procedural",
            "keep_case_candidate": False,
            "normalization_notes": "Reg.17 Art.11(5), likely information-request/procedural",
        }

    if "art.14(3)" in low or "art. 14(3)" in low:
        return {
            "normalized_decision_type": "Procedural Decision",
            "decision_family": "Procedural",
            "keep_case_candidate": False,
            "normalization_notes": "Reg.17 Art.14(3), likely inspection/procedural",
        }

    if "art.15(1)" in low or "art. 15(1)" in low:
        return {
            "normalized_decision_type": "Decision imposing fines",
            "decision_family": "Procedural / Fine",
            "keep_case_candidate": True,
            "normalization_notes": "Reg.17 Art.15(1), fine-related decision; review if only procedural",
        }

    if "art.15(6)" in low or "art. 15(6)" in low:
        return {
            "normalized_decision_type": "Procedural Decision",
            "decision_family": "Procedural",
            "keep_case_candidate": False,
            "normalization_notes": "Reg.17 Art.15(6), provisional/procedural; review",
        }

    if "art.16" in low or "art. 16" in low:
        return {
            "normalized_decision_type": "Periodic Penalty Payment Decision",
            "decision_family": "Procedural / Fine",
            "keep_case_candidate": True,
            "normalization_notes": "Reg.17 Art.16, penalty-payment/fine-related; review",
        }

    if "art.19" in low or "art. 19" in low:
        return {
            "normalized_decision_type": "Procedural Decision",
            "decision_family": "Procedural",
            "keep_case_candidate": False,
            "normalization_notes": "Reg.4056/86 Art.19, likely procedural; review",
        }

    return {
        "normalized_decision_type": "Other Final Decision",
        "decision_family": "Other / Review",
        "keep_case_candidate": pd.NA,
        "normalization_notes": f"unmapped legacy decision_type: {raw}",
    }


normalization_df = pd.DataFrame(web_export["decision_type"].apply(normalize_legacy_decision_type).tolist())
web_export = pd.concat([web_export.drop(columns=[c for c in normalization_df.columns if c in web_export.columns], errors="ignore"), normalization_df], axis=1)

# Re-save the enriched scrape manifest.
save_dataframe(web_export, WEB_SCRAPE_MANIFEST_CSV, WEB_SCRAPE_MANIFEST_PARQUET)

# Also save the mapping actually observed in this scrape, which is useful for manual review.
legacy_decision_type_mapping = (
    web_export
    .groupby(
        ["decision_type", "normalized_decision_type", "decision_family", "keep_case_candidate", "normalization_notes"],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
    .sort_values(["normalized_decision_type", "decision_type"], na_position="last")
)
save_dataframe(
    legacy_decision_type_mapping,
    COM_LEGACY_OUTPUT_DIR / "legacy_decision_type_mapping_observed.csv",
    COM_LEGACY_OUTPUT_DIR / "legacy_decision_type_mapping_observed.parquet",
)

print("Normalized legacy decision categories:")
display(legacy_decision_type_mapping)


Normalized legacy decision categories:


,decision_type,normalized_decision_type,decision_family,keep_case_candidate,normalization_notes,rows
1,Amendment to decision,Amending Decision,Amendment,True,legacy amendment to a decision,5
5,Art.15(1) Reg.17,Decision imposing fines,Procedural / Fine,True,"Reg.17 Art.15(1), fine-related decision; revie...",9
8,Art.15(1) Reg.17 Page : 0001,Decision imposing fines,Procedural / Fine,True,"Reg.17 Art.15(1), fine-related decision; revie...",1
16,Exemption,Exemption Decision,Final Decision,True,legacy exemption/non-opposition wording,46
19,Exemption Art. 81(3) [ex 85(3)],Exemption Decision,Final Decision,True,legacy exemption/non-opposition wording,2
20,Exemption Art.12 Reg.4056/86,Exemption Decision,Final Decision,True,legacy exemption/non-opposition wording,5
21,Exemption with conditions and obligations,Exemption Decision,Final Decision,True,legacy exemption/non-opposition wording,81
35,Non-opposition Reg.4087/88,Exemption Decision,Final Decision,True,legacy exemption/non-opposition wording,1
36,Non-opposition Reg.870/95,Exemption Decision,Final Decision,True,legacy exemption/non-opposition wording,9
37,Non-opposition decision R. 3975/87,Exemption Decision,Final Decision,True,legacy exemption/non-opposition wording,1


## 8. Build CELEX-long / case-number-long tables and URL-level download manifest

`download_url_manifest` is the low-level debug manifest: one row per attempted CELEX/language/format URL.

`download_case_manifest` below is the higher-level coverage view: one row per legacy web-scrape entry, with a flag for whether at least one associated document has been downloaded.


In [56]:

def explode_json_list(df_in: pd.DataFrame, json_col: str, value_col: str) -> pd.DataFrame:
    tmp = df_in.copy()
    tmp[value_col] = tmp[json_col].apply(lambda s: json.loads(s) if isinstance(s, str) and s else [])
    tmp = tmp.explode(value_col, ignore_index=True)
    tmp[value_col] = tmp[value_col].fillna("")
    return tmp


celex_long = explode_json_list(web_export, "celex_complete_list_json", "celex")
celex_long = celex_long[celex_long["celex"].ne("")].drop_duplicates(subset=["source_entry_id", "celex"]).reset_index(drop=True)

case_long = explode_json_list(web_export, "case_number_list_json", "case_number_one")
case_long = case_long[case_long["case_number_one"].ne("")].copy()

save_dataframe(celex_long, COM_LEGACY_OUTPUT_DIR / "web_scrape_manifest_celex_long.csv", COM_LEGACY_OUTPUT_DIR / "web_scrape_manifest_celex_long.parquet")
save_dataframe(case_long, COM_LEGACY_OUTPUT_DIR / "web_scrape_manifest_case_numbers_long.csv", COM_LEGACY_OUTPUT_DIR / "web_scrape_manifest_case_numbers_long.parquet")

print(f"CELEX-long rows: {len(celex_long):,}")
print(f"Case-number-long rows: {len(case_long):,}")


CELEX-long rows: 428
Case-number-long rows: 619


In [57]:

def build_eurlex_url(celex: str, language: str, file_format: str) -> str:
    language = language.upper()
    file_format = file_format.lower()
    if file_format == "html":
        return f"https://eur-lex.europa.eu/legal-content/{language}/TXT/HTML/?uri=CELEX:{celex}"
    if file_format == "pdf":
        return f"https://eur-lex.europa.eu/legal-content/{language}/TXT/PDF/?uri=CELEX:{celex}"
    raise ValueError(f"Unsupported file format: {file_format}")


def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")


def infer_file_format_from_url(url: str) -> str:
    path = urlparse(str(url or "")).path.lower()
    if path.endswith(".pdf") or ".pdf" in path:
        return "pdf"
    return "html"


def infer_language_from_url(url: str, default: str = "EN") -> str:
    text = str(url or "")
    m = re.search(r"[?&]lg=([A-Za-z]{2})\b", text)
    if m:
        return m.group(1).upper()
    m = re.search(r"/([a-z]{2})/[^/]*$", urlparse(text).path, flags=re.I)
    if m:
        return m.group(1).upper()
    m = re.search(r"[_-]([a-z]{2})(?:\.[A-Za-z0-9]+)?$", urlparse(text).path, flags=re.I)
    if m:
        return m.group(1).upper()
    return default.upper()


def output_path_for_celex_download(celex: str, language: str, file_format: str) -> Path:
    suffix = "html" if file_format.lower() == "html" else "pdf"
    folder = COM_LEGACY_FILE_HTML_DIR if suffix == "html" else COM_LEGACY_FILE_PDF_DIR
    return folder / f"{safe_filename(celex)}_{language.upper()}.{suffix}"


def output_path_for_direct_download(source_entry_id: str, url: str, file_format: str) -> Path:
    """
    Output path for direct legacy fallback links.

    Important: keep the source file's own basename as the filename.
    Do NOT prefix it with source_entry_id. This preserves the old/current naming
    convention and prevents redownloading files that already exist as, e.g.,
    iv34801_en.pdf, nail.pdf, audi.pdf, or iv36628.html.
    """
    suffix = "html" if file_format.lower() == "html" else "pdf"
    folder = COM_LEGACY_FILE_HTML_DIR if suffix == "html" else COM_LEGACY_FILE_PDF_DIR
    basename = Path(urlparse(str(url or "")).path).name
    if not basename:
        # Only hash if the link truly has no filename. This should be rare.
        basename = hashlib.sha1(str(url).encode("utf-8")).hexdigest()[:12] + "." + suffix
    if "." not in basename:
        basename = basename + "." + suffix
    return folder / safe_filename(basename)


def output_path_for_download_row(row: pd.Series) -> Path:
    source_type = str(row.get("download_source_type", "celex_eurlex")).strip().lower()
    if source_type == "celex_eurlex":
        return output_path_for_celex_download(row.get("celex", ""), row.get("language", "EN"), row.get("file_format", "html"))
    return output_path_for_direct_download(row.get("source_entry_id", ""), row.get("download_url", ""), row.get("file_format", "html"))


def find_existing_download_path_for_row(row: pd.Series) -> str:
    """
    Check canonical download folders for an already-downloaded file.

    For CELEX rows this checks the canonical CELEX_language filename.
    For direct legacy links this checks whether the original PDF/HTML basename
    from the source page already exists in the legacy files folder.
    """
    expected = output_path_for_download_row(row)
    if expected.exists() and expected.stat().st_size > 0:
        return str(expected)

    source_type = str(row.get("download_source_type", "celex_eurlex")).strip().lower()
    file_format = str(row.get("file_format", "html")).lower().strip()
    suffix = "html" if file_format == "html" else "pdf"
    folder = COM_LEGACY_FILE_HTML_DIR if suffix == "html" else COM_LEGACY_FILE_PDF_DIR

    if source_type == "celex_eurlex":
        celex = safe_filename(str(row.get("celex", "")).strip())
        language = str(row.get("language", "EN")).upper().strip()
        patterns = [f"{celex}_{language}.{suffix}", f"{celex}_*.{suffix}"]
    else:
        basename = Path(urlparse(str(row.get("download_url", ""))).path).name
        patterns = [safe_filename(basename)] if basename else []

    for pattern in patterns:
        for candidate in folder.rglob(pattern):
            if candidate.exists() and candidate.stat().st_size > 0:
                return str(candidate)
    return ""


BASE_DOWNLOAD_COLUMNS = [
    "source_entry_id",
    "source_year",
    "title",
    "decision_date",
    "decision_type",
    "normalized_decision_type",
    "decision_family",
    "keep_case_candidate",
    "official_journal_raw",
    "case_numbers",
    "celex",
    "source_url",
    "download_id",
    "download_source_type",
    "link_label",
    "language",
    "file_format",
    "download_url",
    "download_success",
    "download_status",
    "download_path",
    "http_status",
    "error",
]

DOWNLOAD_RESULT_COLUMNS = [
    "download_success",
    "download_status",
    "download_path",
    "http_status",
    "error",
]


def ensure_download_manifest_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in BASE_DOWNLOAD_COLUMNS:
        if col not in out.columns:
            out[col] = pd.NA
    out = out[BASE_DOWNLOAD_COLUMNS]

    status = out["download_status"].astype("string")
    missing_status = status.isna() | status.str.strip().isin(["", "nan", "none", "<NA>", "<na>"])
    out.loc[missing_status, "download_status"] = "pending"
    out["download_path"] = out["download_path"].fillna("")
    out["error"] = out["error"].fillna("")
    out["download_source_type"] = out["download_source_type"].fillna("celex_eurlex")
    out["link_label"] = out["link_label"].fillna("")
    return out


def refresh_existing_file_status(df: pd.DataFrame, overwrite: bool = False) -> pd.DataFrame:
    out = ensure_download_manifest_columns(df)
    for idx, row in out.iterrows():
        existing_path = find_existing_download_path_for_row(row)
        if existing_path and not overwrite:
            out.at[idx, "download_success"] = True
            out.at[idx, "download_status"] = "already_downloaded"
            out.at[idx, "download_path"] = existing_path
            out.at[idx, "http_status"] = pd.NA
            out.at[idx, "error"] = ""
    return ensure_download_manifest_columns(out)


def _json_list_from_export_row(row: pd.Series, col: str) -> list[str]:
    value = row.get(col, "")
    if isinstance(value, list):
        return _dedupe_keep_order(value)
    if pd.isna(value) if not isinstance(value, (list, dict)) else False:
        return []
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return _dedupe_keep_order(parsed)
        except Exception:
            return _dedupe_keep_order([x.strip() for x in value.split(";")])
    return []


def build_base_download_url_manifest(celex_df: pd.DataFrame, web_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    # 1) CELEX/EUR-Lex candidates.
    for _, row in celex_df.iterrows():
        celex = str(row.get("celex", "")).strip()
        if not celex:
            continue
        base = {
            "source_entry_id": row.get("source_entry_id", ""),
            "source_year": row.get("source_year", ""),
            "title": row.get("title", ""),
            "decision_date": row.get("decision_date", ""),
            "decision_type": row.get("decision_type", ""),
            "normalized_decision_type": row.get("normalized_decision_type", ""),
            "decision_family": row.get("decision_family", ""),
            "keep_case_candidate": row.get("keep_case_candidate", pd.NA),
            "official_journal_raw": row.get("official_journal_raw", ""),
            "case_numbers": row.get("case_numbers", ""),
            "celex": celex,
            "source_url": row.get("source_url", ""),
        }
        for language, file_format in DOWNLOAD_ATTEMPTS:
            download_id = f"{base['source_entry_id']}__celex__{base['celex']}__{language}__{file_format}"
            rows.append({
                **base,
                "download_id": download_id,
                "download_source_type": "celex_eurlex",
                "link_label": "CELEX EUR-Lex",
                "language": language,
                "file_format": file_format,
                "download_url": build_eurlex_url(base["celex"], language, file_format),
                "download_success": pd.NA,
                "download_status": "pending",
                "download_path": "",
                "http_status": pd.NA,
                "error": "",
            })

    # 2) Direct legacy fallback links: full text and OJ external links.
    # Press releases remain metadata only; they are not decision-document candidates.
    for _, row in web_df.iterrows():
        direct_candidates = []
        for url in _json_list_from_export_row(row, "full_text_links_list_json"):
            direct_candidates.append(("full_text", "Legacy full text", url))
        for url in _json_list_from_export_row(row, "oj_external_links_list_json"):
            direct_candidates.append(("oj_external", "Legacy OJ external", url))

        for source_type, label, url in direct_candidates:
            if not url:
                continue
            file_format = infer_file_format_from_url(url)
            language = infer_language_from_url(url, default="EN")
            download_id = f"{row.get('source_entry_id','')}__{source_type}__{hashlib.sha1(url.encode('utf-8')).hexdigest()[:12]}__{language}__{file_format}"
            rows.append({
                "source_entry_id": row.get("source_entry_id", ""),
                "source_year": row.get("source_year", ""),
                "title": row.get("title", ""),
                "decision_date": row.get("decision_date", ""),
                "decision_type": row.get("decision_type", ""),
                "normalized_decision_type": row.get("normalized_decision_type", ""),
                "decision_family": row.get("decision_family", ""),
                "keep_case_candidate": row.get("keep_case_candidate", pd.NA),
                "official_journal_raw": row.get("official_journal_raw", ""),
                "case_numbers": row.get("case_numbers", ""),
                "celex": row.get("celex_primary", ""),
                "source_url": row.get("source_url", ""),
                "download_id": download_id,
                "download_source_type": source_type,
                "link_label": label,
                "language": language,
                "file_format": file_format,
                "download_url": url,
                "download_success": pd.NA,
                "download_status": "pending",
                "download_path": "",
                "http_status": pd.NA,
                "error": "",
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["download_id"]).reset_index(drop=True)
    return refresh_existing_file_status(df, overwrite=OVERWRITE_EXISTING_FILES)


base_download_url_manifest = build_base_download_url_manifest(celex_long, web_export)
print(f"Base download URL manifest rows: {len(base_download_url_manifest):,}")
print(base_download_url_manifest["download_status"].fillna("MISSING").value_counts(dropna=False).head(20))
if "download_source_type" in base_download_url_manifest.columns:
    print(base_download_url_manifest["download_source_type"].fillna("MISSING").value_counts(dropna=False))
base_download_url_manifest.head()


Base download URL manifest rows: 1,781
download_status
already_downloaded    1777
pending                  4
Name: count, dtype: int64
download_source_type
celex_eurlex    1712
oj_external       63
full_text          6
Name: count, dtype: int64


,source_entry_id,source_year,title,decision_date,decision_type,normalized_decision_type,decision_family,keep_case_candidate,official_journal_raw,case_numbers,...,download_source_type,link_label,language,file_format,download_url,download_success,download_status,download_path,http_status,error
0,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,EN,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,<NA>,
1,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,EN,pdf,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,<NA>,
2,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,DE,html,https://eur-lex.europa.eu/legal-content/DE/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,<NA>,
3,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,DE,pdf,https://eur-lex.europa.eu/legal-content/DE/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,<NA>,
4,com_legacy_1964_line_5,1964,Grundig-Consten,23.09.1964,Infringement Art.81 [ex 85],Prohibition Decision,Final Decision,True,L - 20/10/1964 Page : 2545,IV/3344; IV/4,...,celex_eurlex,CELEX EUR-Lex,EN,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,<NA>,


In [58]:

DOWNLOAD_URL_MANIFEST_CSV = COM_LEGACY_OUTPUT_DIR / "download_url_manifest.csv"
DOWNLOAD_URL_MANIFEST_PARQUET = COM_LEGACY_OUTPUT_DIR / "download_url_manifest.parquet"

# Backward compatibility: if an old notebook run produced download_manifest.csv,
# read it once and merge the progress into the new URL-level manifest.
OLD_DOWNLOAD_MANIFEST_CSV = COM_LEGACY_OUTPUT_DIR / "download_manifest.csv"


def merge_existing_download_progress(base_df: pd.DataFrame, existing_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge previously saved download statuses into the freshly built URL manifest.
    File-existence checks are re-applied afterwards, so actual files on disk win.
    """
    base = ensure_download_manifest_columns(base_df)
    existing = existing_df.copy()

    if base.empty:
        return base

    if existing.empty or "download_id" not in existing.columns:
        return refresh_existing_file_status(base, overwrite=OVERWRITE_EXISTING_FILES)

    # Old manifests used 'already_exists'. Normalize that into the clearer new status.
    if "download_status" in existing.columns:
        existing["download_status"] = existing["download_status"].replace({"already_exists": "already_downloaded"})

    existing = ensure_download_manifest_columns(existing)
    existing = existing.drop_duplicates(subset=["download_id"], keep="last")
    progress_cols = [c for c in DOWNLOAD_RESULT_COLUMNS if c in existing.columns]
    merge_cols = ["download_id", *progress_cols]

    merged = base.merge(
        existing[merge_cols],
        on="download_id",
        how="left",
        suffixes=("", "_old"),
    )

    for col in progress_cols:
        old_col = f"{col}_old"
        if old_col in merged.columns:
            merged[col] = merged[old_col].combine_first(merged[col])
            merged = merged.drop(columns=[old_col])

    return refresh_existing_file_status(merged, overwrite=OVERWRITE_EXISTING_FILES)


if DOWNLOAD_URL_MANIFEST_CSV.exists():
    print(f"Loading existing URL download manifest: {DOWNLOAD_URL_MANIFEST_CSV}")
    existing_download_manifest = pd.read_csv(DOWNLOAD_URL_MANIFEST_CSV, low_memory=False)
    download_url_manifest = merge_existing_download_progress(base_download_url_manifest, existing_download_manifest)
elif OLD_DOWNLOAD_MANIFEST_CSV.exists():
    print(f"Loading old download manifest for one-time progress migration: {OLD_DOWNLOAD_MANIFEST_CSV}")
    existing_download_manifest = pd.read_csv(OLD_DOWNLOAD_MANIFEST_CSV, low_memory=False)
    download_url_manifest = merge_existing_download_progress(base_download_url_manifest, existing_download_manifest)
else:
    print("Creating new URL download manifest")
    download_url_manifest = refresh_existing_file_status(base_download_url_manifest, overwrite=OVERWRITE_EXISTING_FILES)

save_dataframe(download_url_manifest, DOWNLOAD_URL_MANIFEST_CSV, DOWNLOAD_URL_MANIFEST_PARQUET)
print(f"Download URL manifest rows: {len(download_url_manifest):,}")
print(download_url_manifest["download_status"].fillna("MISSING").value_counts(dropna=False).head(20))
download_url_manifest.head()


Loading existing URL download manifest: /home/edik/projects/eccjeu/output/com_legacy/download_url_manifest.csv


/tmp/ipykernel_122185/3759337628.py:42: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  merged[col] = merged[old_col].combine_first(merged[col])
/tmp/ipykernel_122185/3759337628.py:42: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  merged[col] = merged[old_col].combine_first(merged[col])


Download URL manifest rows: 1,781
download_status
already_downloaded           1777
failed_http_404_not_found       4
Name: count, dtype: int64


,source_entry_id,source_year,title,decision_date,decision_type,normalized_decision_type,decision_family,keep_case_candidate,official_journal_raw,case_numbers,...,download_source_type,link_label,language,file_format,download_url,download_success,download_status,download_path,http_status,error
0,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,EN,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,NaN,
1,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,EN,pdf,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,NaN,
2,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,DE,html,https://eur-lex.europa.eu/legal-content/DE/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,NaN,
3,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,L - 31/10/1964 Page : 2761,IV/71,...,celex_eurlex,CELEX EUR-Lex,DE,pdf,https://eur-lex.europa.eu/legal-content/DE/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,NaN,
4,com_legacy_1964_line_5,1964,Grundig-Consten,23.09.1964,Infringement Art.81 [ex 85],Prohibition Decision,Final Decision,True,L - 20/10/1964 Page : 2545,IV/3344; IV/4,...,celex_eurlex,CELEX EUR-Lex,EN,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,/home/edik/projects/eccjeu/data/raw/com_legacy...,NaN,


## 9. Build case/source-entry download coverage manifest

`download_case_manifest` is the source-entry-level view. It is the one to inspect when you want to know: *for this scraped legacy entry / CELEX-bearing row, did at least one document download successfully?*


In [59]:

DOWNLOAD_CASE_MANIFEST_CSV = COM_LEGACY_OUTPUT_DIR / "download_case_manifest.csv"
DOWNLOAD_CASE_MANIFEST_PARQUET = COM_LEGACY_OUTPUT_DIR / "download_case_manifest.parquet"


def build_download_case_manifest(web_df: pd.DataFrame, url_df: pd.DataFrame) -> pd.DataFrame:
    web_cols = [
        "source_entry_id",
        "source_year",
        "title",
        "decision_date",
        "decision_type",
        "normalized_decision_type",
        "decision_family",
        "keep_case_candidate",
        "official_journal_raw",
        "case_numbers",
        "case_number_primary",
        "celex_primary",
        "source_url",
        "preferred_download_source",
        "has_any_download_link",
        "has_full_text_link",
        "has_oj_external_link",
        "full_text_links",
        "oj_external_links",
        "press_release_links",
        "needs_manual_review",
    ]
    base = web_df[[c for c in web_cols if c in web_df.columns]].copy()

    if url_df.empty:
        base["celex_count"] = 0
        base["download_url_count"] = 0
        base["downloaded_url_count"] = 0
        base["failed_url_count"] = 0
        base["pending_url_count"] = 0
        base["has_any_download"] = False
        base["download_case_status"] = "no_celex"
        base["downloaded_paths"] = ""
        base["downloaded_celex"] = ""
        return base

    tmp = ensure_download_manifest_columns(url_df)
    status = tmp["download_status"].astype("string").str.lower().fillna("pending")
    success = tmp["download_success"].fillna(False).astype(bool) | status.isin(["downloaded", "already_downloaded", "already_exists"])

    tmp = tmp.assign(
        _success=success,
        _failed=status.str.startswith("failed") | status.isin(["not_found"]),
        _pending=status.isin(["pending", "", "nan", "none", "<na>"]),
    )

    agg = (
        tmp.groupby("source_entry_id", dropna=False)
        .agg(
            celex_count=("celex", lambda s: s.dropna().astype(str).str.strip().replace("", pd.NA).dropna().nunique()),
            celex_list=("celex", lambda s: "; ".join(sorted(set(x for x in s.dropna().astype(str) if x.strip())))),
            download_source_types=("download_source_type", lambda s: "; ".join(sorted(set(x for x in s.dropna().astype(str) if x.strip())))),
            direct_download_url_count=("download_source_type", lambda s: int(s.dropna().astype(str).str.lower().ne("celex_eurlex").sum())),
            download_url_count=("download_id", "count"),
            downloaded_url_count=("_success", "sum"),
            failed_url_count=("_failed", "sum"),
            pending_url_count=("_pending", "sum"),
            downloaded_paths=("download_path", lambda s: "; ".join(sorted(set(x for x in s.dropna().astype(str) if x.strip())))),
            downloaded_celex=("celex", lambda s: "; ".join(sorted(set(
                str(tmp.loc[i, "celex"]) for i in s.index
                if bool(tmp.loc[i, "_success"]) and str(tmp.loc[i, "celex"]).strip()
            )))),
        )
        .reset_index()
    )

    out = base.merge(agg, on="source_entry_id", how="left")
    for col in ["celex_count", "direct_download_url_count", "download_url_count", "downloaded_url_count", "failed_url_count", "pending_url_count"]:
        if col in out.columns:
            out[col] = out[col].fillna(0).astype(int)
    for col in ["celex_list", "download_source_types", "downloaded_paths", "downloaded_celex"]:
        if col in out.columns:
            out[col] = out[col].fillna("")

    out["has_any_download"] = out["downloaded_url_count"].gt(0)

    def status_for_row(row: pd.Series) -> str:
        if row["downloaded_url_count"] > 0:
            return "downloaded"
        if row["pending_url_count"] > 0:
            return "pending"
        if row["failed_url_count"] > 0:
            return "failed"
        if row["download_url_count"] == 0:
            return "no_download_candidate"
        if row["celex_count"] == 0:
            return "no_celex_with_direct_link"
        return "unknown"

    out["download_case_status"] = out.apply(status_for_row, axis=1)

    # Put the coverage fields near the front.
    front = [
        "source_entry_id",
        "source_year",
        "title",
        "decision_date",
        "decision_type",
        "normalized_decision_type",
        "decision_family",
        "keep_case_candidate",
        "case_numbers",
        "case_number_primary",
        "celex_primary",
        "celex_count",
        "celex_list",
        "has_any_download",
        "download_case_status",
        "download_source_types",
        "direct_download_url_count",
        "download_url_count",
        "downloaded_url_count",
        "failed_url_count",
        "pending_url_count",
        "downloaded_celex",
        "downloaded_paths",
    ]
    ordered = [c for c in front if c in out.columns] + [c for c in out.columns if c not in front]
    return out[ordered]


download_case_manifest = build_download_case_manifest(web_export, download_url_manifest)
save_dataframe(download_case_manifest, DOWNLOAD_CASE_MANIFEST_CSV, DOWNLOAD_CASE_MANIFEST_PARQUET)

print(f"Download case manifest rows: {len(download_case_manifest):,}")
print(download_case_manifest["download_case_status"].fillna("MISSING").value_counts(dropna=False))
download_case_manifest.head()


Download case manifest rows: 560
download_case_status
downloaded               432
no_download_candidate    127
failed                     1
Name: count, dtype: int64


,source_entry_id,source_year,title,decision_date,decision_type,normalized_decision_type,decision_family,keep_case_candidate,case_numbers,case_number_primary,...,official_journal_raw,source_url,preferred_download_source,has_any_download_link,has_full_text_link,has_oj_external_link,full_text_links,oj_external_links,press_release_links,needs_manual_review
0,com_legacy_1964_line_2,1964,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,IV/71,IV/71,...,L - 31/10/1964 Page : 2761,http://ec.europa.eu/competition/antitrust/clos...,celex,True,False,False,,,,False
1,com_legacy_1964_line_5,1964,Grundig-Consten,23.09.1964,Infringement Art.81 [ex 85],Prohibition Decision,Final Decision,True,IV/3344; IV/4,IV/3344,...,L - 20/10/1964 Page : 2545,http://ec.europa.eu/competition/antitrust/clos...,celex,True,False,False,,,,False
2,com_legacy_1964_line_8,1964,Nicholas Freres + Vitapro,30.07.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,IV/95,IV/95,...,L - 26/08/1964 Page : 2287,http://ec.europa.eu/competition/antitrust/clos...,celex,True,False,False,,,,False
3,com_legacy_1964_line_11,1964,Bendix + Mertens and Straat,01.06.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,IV/12868,IV/12868,...,L - 10/06/1964 Page : 1426,http://ec.europa.eu/competition/antitrust/clos...,celex,True,False,False,,,,False
4,com_legacy_1964_line_14,1964,Grosfillex + Fillistorf,11.03.1964,Negative clearance Art.81(1) [ex 85(1)],Negative Clearance Decision,Final Decision,True,IV/61,IV/61,...,L - 9/04/1964 Page : 915,http://ec.europa.eu/competition/antitrust/clos...,celex,True,False,True,,http://ec.europa.eu/competition/search.html; h...,,False


## 10. Optional downloader

This cell only downloads files when `DOWNLOAD_FILES = True`.

The downloader is resume-safe:
- it first checks whether the expected file is already present in the source/download folder;
- such rows are marked `already_downloaded`;
- rows still marked `pending` are attempted;
- previous `failed_http_202_accepted` rows are retried;
- terminal failures such as `failed_http_400_bad_request` / `failed_http_404_not_found` are skipped unless manually reset.


In [60]:

def is_probably_valid_download(content: bytes, text: str, identifier: str, file_format: str, content_type: str = "", source_type: str = "celex_eurlex") -> bool:
    """Lightweight validation so we do not save error pages as real documents."""
    file_format = file_format.lower()
    source_type = str(source_type or "").lower()

    if file_format == "pdf":
        return (
            len(content) > 1000
            and (
                content.startswith(b"%PDF")
                or "application/pdf" in str(content_type).lower()
            )
        )

    low = (text or "")[:15000].lower()
    positive_terms = ["eur-lex", "commission", "competition", "official journal", "article 81", "article 82", "ex 85", "ex 86"]
    if source_type == "celex_eurlex" and identifier:
        positive_terms.append(str(identifier).lower())

    return (
        len(text or "") > 500
        and any(term in low for term in positive_terms)
        and "request could not be satisfied" not in low
        and "not found" not in low
        and "javascript is disabled" not in low
        and "verify that you're not a robot" not in low
    )


def failed_status_for_http_status(http_status) -> str:
    try:
        if pd.isna(http_status):
            return "failed_all_attempts"
    except Exception:
        pass
    try:
        code = int(http_status)
    except Exception:
        return "failed_all_attempts"
    if code == 202:
        return "failed_http_202_accepted"
    if code == 400:
        return "failed_http_400_bad_request"
    if code == 404:
        return "failed_http_404_not_found"
    if code >= 500:
        return f"failed_http_{code}_server_error"
    return "failed_all_attempts"


def should_attempt_download_row(row: pd.Series) -> bool:
    status = str(row.get("download_status", "")).strip().lower()
    error = str(row.get("error", "") or "")

    if status in ["", "pending", "nan", "none", "<na>"]:
        return True

    # Retry EUR-Lex 202 accepted/not-ready responses.
    if status == "failed_http_202_accepted":
        return True

    # Retry old v5/v6 rows where Playwright saw a browser download but the
    # previous code did not capture/save it correctly.
    if status == "failed_playwright_download_event":
        return True
    if status == "failed_all_attempts" and "Download is starting" in error:
        return True

    return False


@contextlib.contextmanager
def suppress_stderr():
    try:
        stderr_fd = sys.stderr.fileno()
    except Exception:
        yield
        return
    saved_stderr_fd = os.dup(stderr_fd)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), stderr_fd)
            yield
    finally:
        os.dup2(saved_stderr_fd, stderr_fd)
        os.close(saved_stderr_fd)


async def download_one_row_playwright(row: pd.Series, page, overwrite: bool = False) -> dict:
    """
    Download one CELEX/EUR-Lex or direct legacy full-text/OJ URL.

    Existing files are detected first and returned as already_downloaded.

    v6 patch:
    - PDF URLs may trigger a real browser download event instead of returning a normal
      navigation response. Handle that with page.expect_download() and save_as().
    - Keep the existing naming convention: CELEX files use CELEX_language.format;
      direct fallback files use the source URL basename only.
    """
    source_type = str(row.get("download_source_type", "celex_eurlex")).strip().lower()
    celex = str(row.get("celex", "")).strip()
    language = str(row.get("language", "EN")).upper().strip()
    file_format = str(row.get("file_format", "html")).lower().strip()
    url = str(row.get("download_url", "")).strip()
    identifier = celex or str(row.get("source_entry_id", ""))

    existing_path = find_existing_download_path_for_row(row)
    if existing_path and not overwrite:
        return {
            "download_success": True,
            "download_status": "already_downloaded",
            "download_path": existing_path,
            "http_status": pd.NA,
            "error": "",
        }

    out_path = output_path_for_download_row(row)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # PDF downloads are best handled as direct HTTP requests first.
    # This avoids Playwright's "Page.goto: Download is starting" browser event.
    # If the request route fails, we still fall back to expect_download().
    if file_format == "pdf":
        try:
            api_response = await page.context.request.get(url, timeout=DOWNLOAD_TIMEOUT_SECONDS * 1000)
            http_status = api_response.status
            content_type = api_response.headers.get("content-type", "")
            content = await api_response.body()
            if http_status == 200 and is_probably_valid_download(content, "", identifier, file_format, content_type, source_type):
                out_path.write_bytes(content)
                return {
                    "download_success": True,
                    "download_status": "downloaded",
                    "download_path": str(out_path),
                    "http_status": http_status,
                    "error": "",
                }
            # If direct HTTP gives a clear terminal HTTP status, keep it.
            if http_status in {400, 404}:
                return {
                    "download_success": False,
                    "download_status": failed_status_for_http_status(http_status),
                    "download_path": "",
                    "http_status": http_status,
                    "error": f"status={http_status}; content_type={content_type}; bytes={len(content)}",
                }
            # Otherwise fall through to browser download handling.
        except Exception as e:
            logger.info("Direct PDF request failed for %s; trying browser download fallback: %r", row.get("download_id", ""), e)

        try:
            async with page.expect_download(timeout=PLAYWRIGHT_NAVIGATION_TIMEOUT_MS) as download_info:
                response = await page.goto(
                    url,
                    wait_until="commit",
                    timeout=PLAYWRIGHT_NAVIGATION_TIMEOUT_MS,
                )
            download = await download_info.value
            await download.save_as(str(out_path))
            content = out_path.read_bytes() if out_path.exists() else b""
            if is_probably_valid_download(content, "", identifier, file_format, "application/pdf", source_type):
                return {
                    "download_success": True,
                    "download_status": "downloaded",
                    "download_path": str(out_path),
                    "http_status": response.status if response is not None else pd.NA,
                    "error": "",
                }
            # Do not keep invalid downloaded error pages / placeholders.
            try:
                if out_path.exists():
                    out_path.unlink()
            except Exception:
                pass
            return {
                "download_success": False,
                "download_status": "failed_invalid_pdf_download",
                "download_path": "",
                "http_status": response.status if response is not None else pd.NA,
                "error": f"download event fired but saved file was not a valid PDF; bytes={len(content)}",
            }
        except PlaywrightTimeoutError:
            # No download event. Fall through to response.body() below.
            pass
        except Exception as e:
            # If Playwright still reports a download-starting navigation error, make it
            # an explicit retryable status rather than a generic failed_all_attempts.
            if "Download is starting" in repr(e):
                return {
                    "download_success": False,
                    "download_status": "failed_playwright_download_event",
                    "download_path": "",
                    "http_status": pd.NA,
                    "error": repr(e),
                }
            logger.exception("Playwright PDF download-event path failed for %s", row.get("download_id", ""))
            # Fall through to response.body() fallback.

    try:
        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=PLAYWRIGHT_NAVIGATION_TIMEOUT_MS,
        )

        if PLAYWRIGHT_WAIT_AFTER_GOTO_MS:
            await page.wait_for_timeout(PLAYWRIGHT_WAIT_AFTER_GOTO_MS)

        http_status = response.status if response is not None else pd.NA
        content_type = ""
        if response is not None:
            try:
                content_type = response.headers.get("content-type", "")
            except Exception:
                content_type = ""

        if file_format == "pdf":
            content = await response.body() if response is not None else b""
            if http_status == 200 and is_probably_valid_download(content, "", identifier, file_format, content_type, source_type):
                out_path.write_bytes(content)
                return {
                    "download_success": True,
                    "download_status": "downloaded",
                    "download_path": str(out_path),
                    "http_status": http_status,
                    "error": "",
                }
            return {
                "download_success": False,
                "download_status": failed_status_for_http_status(http_status),
                "download_path": "",
                "http_status": http_status,
                "error": f"status={http_status}; content_type={content_type}; bytes={len(content)}",
            }

        text = await page.content()
        content = text.encode("utf-8", errors="ignore")
        if http_status == 200 and is_probably_valid_download(content, text, identifier, file_format, content_type, source_type):
            out_path.write_text(text, encoding="utf-8", errors="ignore")
            return {
                "download_success": True,
                "download_status": "downloaded",
                "download_path": str(out_path),
                "http_status": http_status,
                "error": "",
            }

        return {
            "download_success": False,
            "download_status": failed_status_for_http_status(http_status),
            "download_path": "",
            "http_status": http_status,
            "error": f"status={http_status}; content_type={content_type}; bytes={len(content)}",
        }

    except Exception as e:
        logger.exception("Playwright download failed for %s", row.get("download_id", ""))
        return {
            "download_success": False,
            "download_status": "failed_all_attempts",
            "download_path": "",
            "http_status": pd.NA,
            "error": repr(e),
        }

def save_download_url_manifest(df: pd.DataFrame) -> None:
    save_dataframe(df, DOWNLOAD_URL_MANIFEST_CSV, DOWNLOAD_URL_MANIFEST_PARQUET)


def save_download_case_manifest_from_url_manifest(url_df: pd.DataFrame) -> pd.DataFrame:
    case_df = build_download_case_manifest(web_export, url_df)
    save_dataframe(case_df, DOWNLOAD_CASE_MANIFEST_CSV, DOWNLOAD_CASE_MANIFEST_PARQUET)
    return case_df


async def run_downloads(download_url_manifest: pd.DataFrame, limit: int | None = None, overwrite: bool = False, save_every: int = 10) -> pd.DataFrame:
    download_url_manifest = refresh_existing_file_status(download_url_manifest, overwrite=overwrite)

    if DOWNLOAD_URL_MANIFEST_CSV.exists():
        current = pd.read_csv(DOWNLOAD_URL_MANIFEST_CSV, low_memory=False)
        df = merge_existing_download_progress(download_url_manifest, current)
    else:
        df = download_url_manifest.copy()

    df = refresh_existing_file_status(df, overwrite=overwrite)
    candidate_mask = df.apply(should_attempt_download_row, axis=1)
    candidate_indices = df.loc[candidate_mask].index.tolist()
    if limit is not None:
        candidate_indices = candidate_indices[:limit]

    print(f"Starting Playwright downloads for {len(candidate_indices):,} pending/retryable files")
    if len(candidate_indices):
        print(df.loc[candidate_indices, "download_status"].fillna("pending").value_counts(dropna=False).head(20))

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-dev-shm-usage", "--no-sandbox", "--disable-gpu"],
        )
        context = await browser.new_context(
            accept_downloads=True,
            user_agent=(
                "Mozilla/5.0 (X11; Linux aarch64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            locale="en-US"
        )
        page = await context.new_page()
        try:
            for counter, idx in enumerate(tqdm(candidate_indices, desc="Downloading COM legacy files"), start=1):
                with suppress_stderr():
                    result = await download_one_row_playwright(df.loc[idx], page, overwrite=overwrite)
                for key, value in result.items():
                    df.at[idx, key] = value
                if counter % save_every == 0:
                    save_download_url_manifest(df)
                    save_download_case_manifest_from_url_manifest(df)
                if DOWNLOAD_SLEEP_SECONDS:
                    await page.wait_for_timeout(int(DOWNLOAD_SLEEP_SECONDS * 1000))
        finally:
            await context.close()
            await browser.close()

    df = refresh_existing_file_status(df, overwrite=overwrite)
    save_download_url_manifest(df)
    save_download_case_manifest_from_url_manifest(df)
    print("Done")
    print(df["download_status"].fillna("MISSING").value_counts(dropna=False).head(20))
    return df


if DOWNLOAD_FILES:
    download_url_manifest = await run_downloads(
        download_url_manifest,
        limit=DOWNLOAD_LIMIT,
        overwrite=OVERWRITE_EXISTING_FILES,
        save_every=SAVE_EVERY,
    )
    download_case_manifest = save_download_case_manifest_from_url_manifest(download_url_manifest)
else:
    print("DOWNLOAD_FILES = False, so no files are downloaded.")
    print("The manifests are ready at:")
    print(DOWNLOAD_URL_MANIFEST_CSV)
    print(DOWNLOAD_CASE_MANIFEST_CSV)


Starting Playwright downloads for 0 pending/retryable files


Done
download_status
already_downloaded           1777
failed_http_404_not_found       4
Name: count, dtype: int64


## 11. Downstream matching notes

Use `source_entry_id` as the stable row identifier for this legacy source. The most useful output tables are:

- `web_scrape_manifest.csv`: enriched scrape manifest, now including `normalized_decision_type`;
- `web_scrape_manifest_celex_long.csv`: one row per legacy source entry / CELEX;
- `web_scrape_manifest_case_numbers_long.csv`: one row per legacy source entry / case number;
- `download_url_manifest.csv`: one row per CELEX/language/format URL attempt;
- `download_case_manifest.csv`: one row per legacy source entry, showing whether at least one associated document was downloaded.

For the final Commission case manifest, treat legacy as a historical metadata source, not just a download source.


Prefinal v5 note: `web_scrape_manifest.csv` now keeps full-text, OJ external, press-release and other link columns on the source row. `download_url_manifest.csv` includes both CELEX/EUR-Lex candidates and direct legacy full-text/OJ fallback links; press releases remain metadata-only.